# History Aware RAG

- In history aware RAG the system looks at the conversation history and rewrites vague or context-dependent questions into clear standalone questions

In [1]:
import os
from dotenv import load_dotenv

print("Exists:", os.path.exists("../openai.env"))

load_dotenv("../openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [2]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [3]:
persistent_directory="db/chroma_db"
embeddings= OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(persist_directory=persistent_directory, embedding_function=embeddings)

In [4]:
model = ChatOpenAI(model = "gpt-4o")

## Store our conversation as messages

In [5]:
chat_history = []

In [21]:
def ask_question(user_question):

    if chat_history:
        messages = [
            SystemMessage(content=(
                "Given the chat history, rewrite the new question to be standalone. "
                "Return only the rewritten question, nothing else."
            ))
        ] + chat_history + [
            HumanMessage(content=f"New Question: {user_question}")
        ]
        search_question = model.invoke(messages).content.strip()
    else:
        search_question = user_question

    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)

    context = "\n".join(f"- {doc.page_content}" for doc in docs)
    combined_input = f"""Based on the following documents, please answer this question: {user_question}
    Documents:
    {context}
    Provide a clear, helpful answer using only the information from these documents.
    If the documents do not contain the answer, say so instead of guessing.
    """

    messages = [
        SystemMessage(content="You are a helpful assistant"),
        HumanMessage(content=combined_input),
    ]
    answer = model.invoke(messages).content

    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    print(answer)

    return answer
        

In [19]:
def start_chat():
    print("Ask me questions! Type 'quit' to exit.")
    
    while True:
        question = input("\nYour question: ")
        
        if question.lower() == 'quit':
            print("Goodbye!")
            break
            
        ask_question(question)

In [20]:
start_chat()

Ask me questions! Type 'quit' to exit.



Your question:  What is photosynthesis


Photosynthesis is a process by which green plants, algae, and some bacteria convert light energy, usually from the sun, into chemical energy stored in glucose. During this process, these organisms use sunlight to transform carbon dioxide and water into glucose and oxygen. The energy captured is then used to fuel their activities and growth. Photosynthesis primarily occurs in the chloroplasts of plant cells, where chlorophyll, a green pigment, plays a crucial role in absorbing light energy. This process is vital for life on Earth as it provides oxygen and serves as the foundation of the food chain. 

Photosynthesis is a system of biological processes by which photopigment-bearing autotrophic organisms, such as most plants, algae, and cyanobacteria, convert light energy—typically from sunlight—into the chemical energy necessary to fuel their metabolism. This process mainly refers to oxygenic photosynthesis, which releases oxygen as a byproduct of water splitting. Photosynthetic organisms


Your question:  quit


Goodbye!
